# 2.2 · Razonamiento con el patrón ReAct
**Módulo 2 · Construye el loop básico del agente** — *Seguridad de agentes con OpenShell y NemoClaw* (Platzi × NVIDIA Deep Learning Institute)

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/uncrayon/curso-seguridad-agentes-nemoclaw/blob/main/clases/2.2-patron-react/2.2-patron-react.ipynb)

**ReAct** (*Reasoning + Acting*, Yao et al., 2022) es el patrón detrás de casi todos los agentes: el modelo **razona**, pide una **acción** (una herramienta), **observa** el resultado y repite hasta tener una respuesta.

En este notebook vas a:

1. Definir tu primera herramienta: `get_current_time`.
2. Ver el campo que gobierna todo el loop: **`finish_reason`**.
3. Construir el loop ReAct **desde cero**, con validación de cada `tool_call` y un tope de pasos.
4. Ver cómo crece `messages[]` en cada paso.
5. 🧯 Provocar tres fallas reales: un `tool_call` mal formado, un loop infinito y `finish_reason="length"`.
6. 🚀 **Proyecto:** darle a Soporte NemoClaw una herramienta nueva, `hora_local`.

*Basado en la lección 01b-react del curso de NVIDIA DLI "Securing Agents with OpenShell and NemoClaw" (Apache-2.0).*

> **¿Sin API key?** Ejecuta todo igual: el notebook detecta que no hay key y usa un **modelo simulado (MOCK)** que imita la forma de las respuestas de la API. Con tu key `nvapi-` (clase 1.2), las mismas celdas llaman al modelo real.

> **¿No programas?** No necesitas leer el código. Presiona ▶ en cada celda, en orden, y lee el texto y lo que aparece debajo (o usa **Entorno de ejecución → Ejecutar todo** en Colab). Las celdas con ⚙️ son de preparación: se ejecutan igual, pero no necesitas abrirlas. Las palabras nuevas están en el [glosario](https://github.com/uncrayon/curso-seguridad-agentes-nemoclaw/blob/main/GLOSARIO.md).

In [ ]:
# @title ⚙️ Configuración (ejecútala siempre primero)
# --- Configuración (ejecútala siempre primero) ---------------------------------
import os, sys, json, subprocess

def _instala(paquete, modulo=None):
    try:
        __import__(modulo or paquete)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", paquete])

_instala("openai")
_instala("python-dotenv", "dotenv")

# 1) Google Colab: guarda tu key en Secrets (icono de llave 🔑) con el nombre NVIDIA_API_KEY
try:
    from google.colab import userdata
    os.environ.setdefault("NVIDIA_API_KEY", userdata.get("NVIDIA_API_KEY") or "")
except Exception:
    pass
# 2) Local: archivo .env en la raíz del repo (copia .env.example) o `export NVIDIA_API_KEY=nvapi-...`
from dotenv import load_dotenv
load_dotenv(); load_dotenv("../../.env")

BASE_URL    = os.environ.get("NVIDIA_BASE_URL", "https://integrate.api.nvidia.com/v1")
MODEL       = os.environ.get("NVIDIA_MODEL", "nvidia/nemotron-3.5-lightning-30b-a3b")
EMBED_MODEL = os.environ.get("NVIDIA_EMBED_MODEL", "nvidia/nemotron-3-embed-1b")
API_KEY     = (os.environ.get("NVIDIA_API_KEY") or "").strip()
USE_MOCK    = not API_KEY

if API_KEY and "integrate.api.nvidia.com" in BASE_URL and not API_KEY.startswith("nvapi-"):
    print("⚠  Tu key no empieza con 'nvapi-'. ¿Copiaste la key completa de build.nvidia.com?")
print("Modo:", "MOCK (sin NVIDIA_API_KEY: respuestas simuladas)" if USE_MOCK else f"EN VIVO · {MODEL}")

In [ ]:
# @title ⚙️ Cliente: modelo real o simulado, con errores explicados en español
# --- Cliente: modelo real o simulado, con errores explicados en español --------
import re, time, math, hashlib
from types import SimpleNamespace as NS

def _msg(content):
    m = NS(content=content, tool_calls=None, role="assistant", reasoning_content=None)
    return NS(choices=[NS(message=m, finish_reason="stop")], usage=NS(prompt_tokens=0, completion_tokens=0, total_tokens=0))

def _call(name, args):
    tc = NS(id="call_" + hashlib.md5((name + json.dumps(args)).encode()).hexdigest()[:8], type="function",
            function=NS(name=name, arguments=json.dumps(args, ensure_ascii=False)))
    m = NS(content=None, tool_calls=[tc], role="assistant", reasoning_content=None)
    return NS(choices=[NS(message=m, finish_reason="tool_calls")], usage=NS(prompt_tokens=0, completion_tokens=0, total_tokens=0))

class MockClient:
    """Imita la FORMA de la API de chat completions. No es inteligente: sigue reglas.
    Cada notebook puede agregar reglas propias con  @regla_mock  (ver más abajo)."""
    reglas = []
    def __init__(self):
        self.chat = NS(completions=NS(create=self._create))
        self.embeddings = NS(create=self._embed)
    def _create(self, model=None, messages=None, tools=None, **kw):
        nombres = [t["function"]["name"] for t in (tools or [])]
        for r in self.reglas:
            out = r(messages, nombres)
            if out is not None:
                return out
        ultimo = messages[-1]
        texto = (ultimo.get("content") or "")
        if ultimo["role"] == "tool":
            return _msg("Según la herramienta: " + str(ultimo["content"]))
        if re.search(r"c[oó]mo me llamo|what is my name", texto, re.I):
            vistos = [re.search(r"(?:me llamo|my name is)\s+(\w+)", m.get("content") or "", re.I)
                      for m in messages[:-1] if m["role"] == "user"]
            vistos = [v.group(1) for v in vistos if v]
            return _msg(f"Te llamas {vistos[-1]}." if vistos else "No sé cómo te llamas: no me lo has dicho en esta conversación.")
        return _msg("(mock) Respuesta simulada. Configura NVIDIA_API_KEY para hablar con el modelo real.")
    def _embed(self, model=None, input=None, **kw):
        textos = [input] if isinstance(input, str) else list(input)
        return NS(data=[NS(embedding=_vector_bolsa(t), index=i) for i, t in enumerate(textos)])

def regla_mock(f):
    """Decorador: agrega una regla al modelo simulado. f(messages, nombres_de_tools) -> respuesta | None"""
    MockClient.reglas.append(f)
    return f

_VACIAS = set("que qué como cómo para por con una uno los las del este esta eso esto hago hace tengo puedo sale mis tus sus the and you what how".split())

def _vector_bolsa(texto, dim=256):
    """Embedding simulado: bolsa de palabras con hashing. Sirve para que RAG funcione offline."""
    v = [0.0] * dim
    for w in re.findall(r"\w+", texto.lower()):
        if len(w) > 2 and w not in _VACIAS:
            w = w[:6]                       # raíz burda: "reanudar" ~ "reanudé"
            v[int(hashlib.md5(w.encode()).hexdigest(), 16) % dim] += 1.0
    n = math.sqrt(sum(x * x for x in v)) or 1.0
    return [x / n for x in v]

def diagnostico(e):
    """Traduce un error de la API a: qué pasó + qué hacer. Es el 'FAQ' en código."""
    status = getattr(e, "status_code", None)
    cuerpo = str(getattr(e, "body", "") or e)[:300]
    tabla = {
        400: "400 · Solicitud inválida: algún parámetro no le gusta al modelo (¿tools en un modelo que no las soporta? ¿max_tokens muy alto?). Lee el cuerpo del error.",
        401: "401 · Key rechazada: revisa que NVIDIA_API_KEY sea la key completa (empieza con nvapi-), sin espacios ni comillas. Si la regeneraste, la vieja ya no sirve.",
        403: "403 · Tu key es válida pero no tiene acceso a ese modelo o se acabaron tus créditos. Revisa tu cuenta en build.nvidia.com o prueba otro modelo.",
        404: "404 · Ese modelo o esa ruta no existen. Revisa MODEL / EMBED_MODEL (copia el ID exacto del catálogo) y que BASE_URL termine en /v1.",
        410: "410 · El modelo fue retirado del endpoint gratuito. Cambia MODEL por uno vigente del catálogo de build.nvidia.com.",
        422: "422 · Argumentos con formato incorrecto (p. ej. falta input_type en embeddings o un esquema de tool inválido).",
        429: "429 · Demasiadas solicitudes por minuto. Espera unos segundos y reintenta; no ejecutes celdas en ráfaga.",
    }
    if status in tabla:
        return f"{tabla[status]}\n   Detalle del servidor: {cuerpo}"
    if status and 500 <= status < 600:
        return f"{status} · Falla del lado del servidor o el modelo está arrancando. Reintenta en 30-60 s; si persiste, prueba otro modelo.\n   Detalle: {cuerpo}"
    nombre = type(e).__name__
    if "Timeout" in nombre:
        return "Timeout · El modelo tardó demasiado. Reintenta, baja max_tokens o usa un modelo más rápido."
    if "Connection" in nombre:
        return "Sin conexión · No se pudo llegar a BASE_URL. Revisa tu internet, VPN/proxy corporativo o el valor de BASE_URL."
    return f"{nombre}: {cuerpo}"

if USE_MOCK:
    client = MockClient()
else:
    from openai import OpenAI
    client = OpenAI(base_url=BASE_URL, api_key=API_KEY, timeout=60, max_retries=2)

def chat(messages, tools=None, model=None, **kw):
    """Una llamada al modelo. Devuelve la respuesta cruda (choices[0].message, finish_reason, usage)."""
    kw.setdefault("max_tokens", 4096)          # los modelos que razonan gastan tokens antes de responder
    if tools:
        kw["tools"] = tools
    try:
        r = client.chat.completions.create(model=model or MODEL, messages=messages, **kw)
    except Exception as e:
        raise RuntimeError(diagnostico(e)) from None
    ch = r.choices[0]
    if ch.finish_reason == "length" and not (ch.message.content or ch.message.tool_calls):
        print("⚠  finish_reason=length sin contenido: el modelo gastó max_tokens razonando. Sube max_tokens.")
    return r

def embed(textos, input_type="passage", model=None):
    """Embeddings. input_type='query' para preguntas, 'passage' para documentos."""
    textos = [textos] if isinstance(textos, str) else list(textos)
    try:
        r = client.embeddings.create(model=model or EMBED_MODEL, input=textos,
                                     extra_body={"input_type": input_type, "truncate": "END"}) if not USE_MOCK \
            else client.embeddings.create(model=model or EMBED_MODEL, input=textos)
    except Exception as e:
        raise RuntimeError(diagnostico(e)) from None
    return [d.embedding for d in r.data]

print("Cliente listo:", "MockClient" if USE_MOCK else BASE_URL)

> **Celda del simulador.** Solo se usa en modo MOCK: le enseña al modelo simulado a comportarse como un modelo real en *este* notebook (pedir herramientas, equivocarse a veces). Con tu key no se usa. Ejecútala y sigue; no hace falta leerla.

In [ ]:
def _ultima_tool(messages):
    """(nombre, contenido) del último mensaje 'tool', buscando su tool_call_id en los turnos del asistente."""
    m = messages[-1]
    if m.get("role") != "tool":
        return None, None
    for prev in reversed(messages):
        for tc in prev.get("tool_calls") or []:
            if tc["id"] == m.get("tool_call_id"):
                return tc["function"]["name"], m.get("content") or ""
    return "?", m.get("content") or ""

def _pregunta(messages):
    return next((m.get("content") or "" for m in reversed(messages) if m.get("role") == "user"), "")

_CIUDADES = ["ciudad de méxico", "cdmx", "guadalajara", "monterrey", "bogotá", "bogota", "lima", "quito",
             "caracas", "la paz", "buenos aires", "montevideo", "são paulo", "sao paulo", "guatemala", "panamá"]
_LARGO = ("ReAct es un patrón en el que el modelo alterna razonamiento y acción. Primero piensa qué le falta para "
          "responder; luego pide una herramienta con argumentos concretos; el harness la ejecuta y le devuelve el "
          "resultado como una observación; el modelo vuelve a razonar con esa nueva información y decide si necesita "
          "otra herramienta o si ya puede responder. Por ejemplo, para saber cuántos minutos faltan para una reunión, "
          "primero lee el reloj y después hace la resta. El patrón se publicó en 2022 y es la base de LangChain, "
          "OpenClaw y casi todos los agentes modernos. Su gran ventaja es que cada paso queda registrado en messages, "
          "así que puedes auditar qué hizo el agente y por qué.")

@regla_mock
def _react(messages, nombres):
    sistema = (messages[0].get("content") or "") if messages[0]["role"] == "system" else ""
    q = _pregunta(messages).lower()
    tool, resultado = _ultima_tool(messages)
    # 1) El loop "terco": pide la hora hasta que los segundos terminen en 00
    if "terminen en 00" in sistema and "get_current_time" in nombres:
        if resultado and resultado.endswith(":00 UTC"):
            return _msg(f"Listo: {resultado}")
        return _call("get_current_time", {})
    # 2) Respuesta después de ejecutar una herramienta
    if tool == "get_current_time" and not resultado.startswith("ERROR"):
        meta = re.search(r"para las (\d{1,2}):(\d{2})", q)
        if meta:
            ahora = int(resultado[11:13]) * 60 + int(resultado[14:16])
            falta = (int(meta.group(1)) * 60 + int(meta.group(2)) - ahora) % (24 * 60)
            return _msg(f"Son las {resultado[11:16]} UTC, así que faltan {falta} minutos para las {meta.group(1)}:{meta.group(2)} UTC.")
        return _msg(f"Son las {resultado[11:19]} UTC del {resultado[:10]} (lo leí con get_current_time).")
    if tool == "hora_local":
        if resultado.startswith("ERROR"):
            return _msg("No tengo la zona horaria de esa ciudad. ¿Me dices una ciudad cercana de la lista?")
        m = re.search(r"(\d{2}:\d{2}) en (.+)$", resultado)
        return _msg(f"Ahora mismo son las {m.group(1)} en {m.group(2)}." if m else "Según la herramienta: " + resultado)
    if tool:
        return _msg("Según la herramienta: " + resultado)
    if messages[-1]["role"] != "user":
        return None
    # 3) ¿Pedir una herramienta?
    if "hora_local" in nombres:
        ciudad = next((c for c in _CIUDADES if c in q), None)
        if ciudad:
            return _call("hora_local", {"ciudad": "CDMX" if ciudad == "cdmx" else ciudad.title()})
    if "get_current_time" in nombres and re.search(r"hora|time|reloj|minutos|fecha", q):
        return _call("get_current_time", {})
    # 4) Respuestas sin herramientas
    if "detalle" in q:
        return _msg(_LARGO)
    if re.search(r"hora|time", q):
        return _msg("No puedo saber la hora actual: no tengo reloj ni acceso a internet; solo conozco mis datos de entrenamiento.")
    if "react" in q:
        return _msg("ReAct es un loop en el que el modelo razona, pide una herramienta, observa el resultado y repite hasta poder responder.")
    if "capital de francia" in q:
        return _msg("La capital de Francia es París.")
    return None

# Imitar finish_reason="length" cuando max_tokens es muy bajo (un modelo real corta la respuesta)
if USE_MOCK and not getattr(client, "_con_tope", False):
    _crear_original = client.chat.completions.create
    def _crear_con_tope(**kw):
        r = _crear_original(**kw)
        c, tope = r.choices[0], kw.get("max_tokens", 4096)
        if c.message.content and len(c.message.content) > tope * 4:     # ~4 caracteres por token
            c.message.content = c.message.content[: tope * 4]
            c.finish_reason = "length"
        return r
    client.chat.completions.create = _crear_con_tope
    client._con_tope = True

print("Reglas del simulador listas:", len(MockClient.reglas))

## 1 · Una herramienta

Una **herramienta** (tool) tiene dos partes:

- la **función** de Python que tu código ejecuta, y
- su **esquema**: nombre, descripción y parámetros en JSON Schema. El modelo **solo ve el esquema**, nunca tu código.

Un modelo no tiene reloj: todo lo que sabe viene de su entrenamiento. `get_current_time` le da una forma de leer la hora real.

In [ ]:
from datetime import datetime, timezone, timedelta

def get_current_time():
    """Hora actual en UTC como 'AAAA-MM-DD HH:MM:SS UTC'."""
    return datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")

TOOL_HORA = {
    "type": "function",
    "function": {
        "name": "get_current_time",
        "description": ("Devuelve la fecha y hora actual en UTC ('AAAA-MM-DD HH:MM:SS UTC'). "
                        "Úsala para cualquier pregunta sobre la hora o la fecha de hoy: tú no tienes reloj."),
        "parameters": {"type": "object", "properties": {}, "required": []},
    },
}
TOOLS = [TOOL_HORA]
REGISTRO = {"get_current_time": get_current_time}      # nombre → función que de verdad se ejecuta

print(REGISTRO["get_current_time"]())

## 2 · `finish_reason`: el campo que decide el siguiente paso

Hacemos dos llamadas: una sin herramientas y otra **ofreciendo** `get_current_time`. La herramienta no se ejecuta todavía: solo miramos qué **pide** el modelo.

In [ ]:
P1 = [{"role": "user", "content": "En una oración: ¿qué es el loop ReAct de un agente?"}]
P2 = [{"role": "user", "content": "¿Qué hora es ahora en UTC? Usa get_current_time para leer el reloj."}]

for etiqueta, msgs, tools in [("sin tools", P1, None), ("con tools", P2, TOOLS)]:
    r = chat(msgs, tools=tools)
    c = r.choices[0]
    print(f"[{etiqueta}] finish_reason = {c.finish_reason!r}")
    print("   content   :", (c.message.content or "")[:150])
    print("   tool_calls:", [(t.function.name, t.function.arguments) for t in (c.message.tool_calls or [])])

| `finish_reason` | Qué significa | Qué hace el loop |
|---|---|---|
| `stop` | El modelo terminó su respuesta | Devolverla y salir |
| `tool_calls` | Pide ejecutar una o más herramientas | Validar, ejecutar, agregar el resultado y volver a llamar |
| `length` | Se acabó `max_tokens` | Cortar con un error claro (subir `max_tokens` o acortar el contexto) |
| `content_filter` | Un filtro de seguridad detuvo la salida | Avisar al usuario |

Si un agente se comporta raro, **mira `finish_reason` primero**.

**✏️ Tu turno.** En `P2`, borra la frase "Usa get_current_time para leer el reloj." y vuelve a ejecutar. ¿El modelo sigue pidiendo la herramienta solo con la descripción?

## 3 · El loop ReAct desde cero

El router es `finish_reason`. Por cada `tool_call` el harness hace tres cosas: **validar** (¿existe la herramienta? ¿los argumentos son JSON válido?), **ejecutar** y **agregar el resultado** como un mensaje con `role: "tool"` y el mismo `tool_call_id`.

Y siempre hay un **tope de pasos**: un loop sin límite es una factura esperando a llegar.

In [ ]:
def a_dict(msg):
    """Convierte la respuesta del asistente en un dict para guardarlo en messages."""
    d = {"role": "assistant", "content": msg.content or ""}
    if msg.tool_calls:
        d["tool_calls"] = [{"id": t.id, "type": "function",
                            "function": {"name": t.function.name, "arguments": t.function.arguments or "{}"}}
                           for t in msg.tool_calls]
    return d

def mostrar(messages):
    """Imprime messages turno por turno y estima cuántos tokens se reenvían en la próxima llamada."""
    total = 0
    for i, m in enumerate(messages):
        if m.get("tool_calls"):
            cuerpo = "pide → " + ", ".join(f'{t["function"]["name"]}({t["function"]["arguments"]})' for t in m["tool_calls"])
        else:
            cuerpo = (m.get("content") or "").replace("\n", " ")
        total += len(json.dumps(m, ensure_ascii=False))
        print(f"{i:2d} · {m['role']:9s} · {cuerpo[:95]}")
    print(f"   ≈ {total // 4} tokens se reenvían en la próxima llamada (regla rápida: ~4 caracteres por token)")


def validar_tool_call(tc, registro):
    """Devuelve (args, None) si la llamada es válida, o (None, mensaje_de_error) si no."""
    nombre = tc.function.name
    if nombre not in registro:
        return None, f"ERROR: la herramienta {nombre!r} no existe. Disponibles: {list(registro)}"
    try:
        args = json.loads(tc.function.arguments or "{}")
    except json.JSONDecodeError as e:
        return None, f"ERROR: los argumentos no son JSON válido ({e.msg}). Reenvía la llamada con JSON correcto."
    if not isinstance(args, dict):
        return None, "ERROR: los argumentos deben ser un objeto JSON {...}."
    return args, None

SISTEMA = ("Usa get_current_time cuando la pregunta necesite la hora actual: tus datos de entrenamiento no tienen reloj. "
           "Responde a partir del resultado de la herramienta.")

def react(pregunta, tools=TOOLS, registro=REGISTRO, sistema=SISTEMA, max_pasos=6, verbose=True, **kw):
    messages = [{"role": "system", "content": sistema}, {"role": "user", "content": pregunta}]
    for paso in range(1, max_pasos + 1):
        r = chat(messages, tools=tools, **kw)
        c = r.choices[0]
        if verbose:
            print(f"[paso {paso}] finish_reason={c.finish_reason!r} · se enviaron {len(messages)} mensajes")
            razon = getattr(c.message, "reasoning_content", None)
            if razon:
                print("   🧠", razon.replace("\n", " ")[:120], "…")
        # --- el router ---------------------------------------------------------
        if c.finish_reason == "length":
            return "⛔ El modelo llegó a su límite de tokens (length). Sube max_tokens o acorta el contexto.", messages
        messages.append(a_dict(c.message))
        if c.finish_reason == "tool_calls" or c.message.tool_calls:   # algunos servidores dicen 'stop' aunque pidan tools
            for tc in c.message.tool_calls:
                args, error = validar_tool_call(tc, registro)
                if error:
                    resultado = error                                 # el error vuelve al modelo como observación
                else:
                    try:
                        resultado = str(registro[tc.function.name](**args))
                    except Exception as e:
                        resultado = f"ERROR al ejecutar {tc.function.name}: {type(e).__name__}: {e}"
                if verbose:
                    print(f"   🔧 {tc.function.name}({tc.function.arguments}) → {resultado[:100]}")
                messages.append({"role": "tool", "tool_call_id": tc.id, "content": resultado})
            continue
        if c.finish_reason == "stop" and c.message.content:
            return c.message.content, messages
        return f"⛔ El modelo no dio respuesta final ni pidió herramientas (finish_reason={c.finish_reason!r}).", messages
    return f"⛔ Tope de {max_pasos} pasos sin respuesta final. Revisa el historial de herramientas.", messages

respuesta, traza = react("¿Qué hora es en UTC?")
print("\n💬", respuesta)

Deberías ver dos pasos: `[paso 1] finish_reason='tool_calls'` con `🔧 get_current_time({})`, y `[paso 2] finish_reason='stop'` con la respuesta.

## 4 · Mira crecer `messages[]`

El historial es la **única memoria** del agente. Cada paso agrega turnos y la próxima llamada reenvía todo.

In [ ]:
mostrar(traza)

Turnos 0 a 3: `system`, `user`, `assistant` (pide la herramienta) y `tool` (el resultado). Ojo: que el modelo **pida** una herramienta no prueba que se ejecutó. La prueba es el mensaje `tool` en el historial.

**✏️ Tu turno.**

1. Pregunta algo que la herramienta no resuelve: `"¿Cuál es la capital de Francia?"`. ¿En qué paso aparece `stop`?
2. Pregunta `"¿Cuántos minutos faltan para las 17:00 UTC?"` y cuenta cuántas veces se llama a `get_current_time`. Si el modelo la llama más de una vez, ¿cómo lo desalentarías desde el system prompt?

In [ ]:
for q in ["¿Cuál es la capital de Francia?", "¿Cuántos minutos faltan para las 17:00 UTC?"]:
    print("👤", q)
    respuesta, t = react(q)
    llamadas = sum(len(m.get("tool_calls", [])) for m in t)
    print(f"💬 {respuesta}\n   (herramientas llamadas: {llamadas})\n")

## 5 · 🧯 Cuando algo falla

### a) Un `tool_call` mal formado

Los modelos a veces mandan JSON roto, inventan nombres de herramientas o mandan argumentos que la función no acepta. Fabricamos esas llamadas a mano y vemos qué hace el harness: **ninguna rompe el loop**, todas se vuelven un mensaje de error que el modelo puede leer y corregir.

In [ ]:
from types import SimpleNamespace as NS

def tc_falso(nombre, argumentos):
    return NS(id="call_falso", type="function", function=NS(name=nombre, arguments=argumentos))

casos = [
    tc_falso("get_current_time", "{zona: UTC"),         # JSON roto
    tc_falso("get_time", "{}"),                         # herramienta inventada
    tc_falso("get_current_time", "[1, 2]"),             # no es un objeto
    tc_falso("get_current_time", '{"zona": "UTC"}'),    # argumento que la función no acepta
]
for tc in casos:
    args, error = validar_tool_call(tc, REGISTRO)
    if not error:
        try:
            REGISTRO[tc.function.name](**args)
        except Exception as e:
            error = f"ERROR al ejecutar {tc.function.name}: {type(e).__name__}: {e}"
    print(f"{tc.function.name}({tc.function.arguments}) →\n   {error}\n")

### b) Un loop infinito

Este system prompt le pide al modelo algo que casi nunca se cumple. Sin `max_pasos`, el agente seguiría llamando al reloj (y gastando créditos) para siempre. Si un notebook tuyo se queda así, detén la celda con ■.

In [ ]:
SISTEMA_TERCO = ("Llama a get_current_time una y otra vez hasta que los segundos terminen en 00. "
                 "Solo entonces responde con la hora.")
respuesta, t = react("¿Qué hora es?", sistema=SISTEMA_TERCO, max_pasos=4)
print("\n💬", respuesta)

Deberías ver `⛔ Tope de 4 pasos sin respuesta final`. Con un modelo real el resultado puede variar (a veces el modelo se rinde antes y responde), pero **el tope es lo que te protege**, no la buena voluntad del modelo.

### c) `finish_reason = "length"`

Si `max_tokens` es muy chico, el modelo se corta a la mitad. Con modelos que razonan, como Nemotron, puede gastarse todo el presupuesto pensando y devolver `content` vacío.

In [ ]:
respuesta, t = react("Explica con mucho detalle qué es ReAct, con un ejemplo.", max_tokens=20)
print("\n💬", respuesta)

La solución: subir `max_tokens` (el helper `chat()` usa 4096 por defecto) o pedir respuestas más cortas. Nunca trates una respuesta cortada como si fuera la final. Más detalles en la [FAQ](../../FAQ.md#elength).

## 6 · Context rot: por qué un loop largo lee peor

Cada paso agrega mensajes. La siguiente llamada es más **larga**, más **lenta**, más **cara** y el modelo **atiende peor** lo que quedó en medio del contexto: es el efecto *lost in the middle* (Liu et al., 2023). A eso se le llama **context rot** (degradación del contexto).

In [ ]:
def tokens_aprox(messages):
    return sum(len(json.dumps(m, ensure_ascii=False)) for m in messages) // 4

base = tokens_aprox(traza)
for rondas in [1, 5, 10, 20]:
    extra = [{"role": "tool", "tool_call_id": "x", "content": "x" * 800}] * (rondas - 1)
    print(f"después de {rondas:2d} rondas de herramientas ≈ {base + tokens_aprox(extra):6d} tokens por llamada")

Tres mitigaciones que vas a construir en el Módulo 3:

- **Planear primero** y ejecutar después (ReWOO, clase 3.1).
- **Recuperar** solo los fragmentos relevantes en vez de pegar documentos enteros (RAG, clase 3.2).
- **Delegar** subtareas a subagentes con su propio contexto, o **resumir** los turnos viejos (clase 3.3).

## 7 · 🚀 Avance del proyecto: Soporte NemoClaw con una herramienta

Los estudiantes del curso están en toda Latinoamérica. Agregamos una herramienta nueva, `hora_local(ciudad)`, para que Soporte NemoClaw responda preguntas como "¿qué hora es en Bogotá?" sin inventar.

Fíjate en lo que **no** cambia: la función `react`. Solo agregas el esquema a la lista de tools y la función al registro.

In [ ]:
# Ciudades sin horario de verano (offset fijo respecto a UTC)
ZONAS = {"ciudad de méxico": -6, "cdmx": -6, "guadalajara": -6, "monterrey": -6, "guatemala": -6,
         "bogotá": -5, "bogota": -5, "lima": -5, "quito": -5, "panamá": -5,
         "caracas": -4, "la paz": -4,
         "buenos aires": -3, "montevideo": -3, "são paulo": -3, "sao paulo": -3}

def hora_local(ciudad):
    clave = ciudad.strip().lower()
    if clave not in ZONAS:
        return f"ERROR: no conozco la zona horaria de {ciudad!r}. Ciudades disponibles: {sorted(ZONAS)}"
    t = datetime.now(timezone.utc) + timedelta(hours=ZONAS[clave])
    return t.strftime("%Y-%m-%d %H:%M") + f" en {ciudad} (UTC{ZONAS[clave]:+d})"

TOOL_HORA_LOCAL = {
    "type": "function",
    "function": {
        "name": "hora_local",
        "description": ("Hora local actual en una ciudad de Latinoamérica. Úsala cuando el estudiante mencione "
                        "su ciudad o pregunte la hora en un lugar específico."),
        "parameters": {
            "type": "object",
            "properties": {"ciudad": {"type": "string", "description": "Nombre de la ciudad, por ejemplo 'Bogotá' o 'CDMX'"}},
            "required": ["ciudad"],
        },
    },
}

SISTEMA_SOPORTE = """Eres "Soporte NemoClaw", el asistente del curso Seguridad de agentes con OpenShell y NemoClaw (Platzi × NVIDIA).
Respondes dudas de estudiantes en español, en máximo 3 oraciones.
Usa las herramientas cuando necesites la hora: nunca la inventes. Si una herramienta devuelve un error, explícalo y pide el dato que falta."""

TOOLS_SOPORTE = [TOOL_HORA, TOOL_HORA_LOCAL]
REGISTRO_SOPORTE = {"get_current_time": get_current_time, "hora_local": hora_local}

respuesta, t = react("Soy de Bogotá. ¿Qué hora es allá ahora mismo?",
                     tools=TOOLS_SOPORTE, registro=REGISTRO_SOPORTE, sistema=SISTEMA_SOPORTE)
print("\n💬", respuesta, "\n")
mostrar(t)

## 🏆 Reto: evitar llamadas redundantes

Agrega una tercera herramienta, `minutos_hasta(hora_utc)`, que reciba una hora como `"17:00"` y devuelva cuántos minutos faltan desde ahora (en UTC).

**Criterio de éxito:** con tu herramienta en `TOOLS_SOPORTE` y `REGISTRO_SOPORTE`, la pregunta `"¿Cuántos minutos faltan para las 17:00 UTC?"` se responde con **una sola** llamada a herramienta (cuéntalas como en la sección 4) y el número coincide con tu cálculo a mano. Pruébalo en vivo.

In [ ]:
# ✏️ Tu solución aquí
def minutos_hasta(hora_utc):
    ...

## Resumen

- ReAct alterna **razonar → actuar → observar** hasta tener una respuesta.
- `finish_reason` es el router: `stop` responde, `tool_calls` ejecuta, `length` corta con un error claro.
- Cada `tool_call` se **valida**, se **ejecuta** y su resultado vuelve como un mensaje `tool` con el mismo `tool_call_id`.
- Siempre hay **tope de pasos**. Y cuanto más largo el historial, peor lee el modelo (context rot).

**Siguiente clase (2.3):** de una herramienta a varias. El contrato, las descripciones, la validación de argumentos y las herramientas reales del proyecto: `buscar_faq` y `estimar_costo_brev`.